# Formula parser: step-by-step tests

Mỗi code cell bên dưới kiểm tra một hàm hoặc một bước xử lý chính.

In [ ]:
from pathlib import Path
import importlib
import pprint
import sys

candidates = [
    Path.cwd(),
    Path.cwd() / "src" / "excel_engine",
    Path.cwd() / "python-engine" / "src" / "excel_engine",
]
module_dir = next(
    (path for path in candidates if (path / "formula_parser.py").exists()),
    None,
)
if module_dir is None:
    raise FileNotFoundError("Cannot find formula_parser.py")

if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

fp = importlib.import_module("formula_parser")

def show(title, value):
    print("\n" + "=" * 88)
    print(title)
    print("-" * 88)
    pprint.pprint(value, width=100, sort_dicts=False)


In [ ]:
raw_text = """
S_V1 = 2 x PI() x W
S_V2 = 2 x H x L_M
S = K x (S_V1 + S_V2) / 1000000
"""

variables = {
    "W": 500.0,
    "H": 300.0,
    "L_M": 2000.0,
    "K": 1.5,
}

cells = {}
references = {
    "W": "B2",
    "H": "B3",
    "L_M": "B4",
    "K": "B5",
}

show("Input fixtures", {"raw_text": raw_text, "variables": variables})


In [ ]:
# 1. Test _normalize_source().
normalized = fp._normalize_source(raw_text)
show("1. _normalize_source(raw_text)", normalized)
assert " x " not in normalized
assert "2*PI()*W" in normalized
assert "2*H*L_M" in normalized


In [ ]:
# 2. Test extract_assignments().
assignments = fp.extract_assignments(raw_text)
show("2. extract_assignments(raw_text)", assignments)
assert [label for label, _ in assignments] == ["S_V1", "S_V2", "S"]


In [ ]:
# 3. Test _tokens().
expression = assignments[0][1]
tokens = fp._tokens(expression)
show("3. _tokens(first expression)", tokens)
assert tokens[-1].kind == "eof"


In [ ]:
# 4. Test parse_formula().
node = fp.parse_formula(expression)
show("4. parse_formula(first expression) -> AST", node)
assert node.kind == "binary"


In [ ]:
# 5. Test evaluate().
value = fp.evaluate(node, variables, cells)
show("5. evaluate(AST, variables, cells)", value)
assert abs(value - (2 * 3.141592653589793 * variables["W"])) < 1e-9


In [ ]:
# 6. Test render_excel().
excel_formula = fp.render_excel(node, references)
show("6. render_excel(AST, references)", "=" + excel_formula)
assert "B2" in excel_formula


In [ ]:
# 7. Test canonical().
canonical_value = fp.canonical(node)
show("7. canonical(AST)", canonical_value)
assert canonical_value


In [ ]:
# 8. Test formula_result().
single_result = fp.formula_result(
    expression=expression,
    variables=variables,
    cells=cells,
    references=references,
)
show("8. formula_result(...)", single_result)
assert "value" in single_result
assert "excel_formula" in single_result


In [ ]:
# 9. Test _assignment_key().
assignment_key = fp._assignment_key("s_v1")
show("9. _assignment_key('s_v1')", assignment_key)
assert assignment_key == "S_V1"


In [ ]:
# 10. Test _with_assignment_values().
scope = fp._with_assignment_values(variables, {"S_V1": 123.45})
show("10. _with_assignment_values(...)", scope)
assert scope["S_V1"] == 123.45


In [ ]:
# 11. Test _expand_assignments().
parsed_definitions = {
    label: fp.parse_formula(expression_text)
    for label, expression_text in assignments
}
expanded_s = fp._expand_assignments(
    parsed_definitions["S"],
    parsed_definitions,
)
show("11. _expand_assignments(S)", expanded_s)
assert expanded_s.kind == "binary"


In [ ]:
# 12. Test formula_block_result(), the complete workflow.
block_result = fp.formula_block_result(
    assignments=assignments,
    variables=variables,
    cells=cells,
    references=references,
    target="S",
)
show("12. formula_block_result(..., target='S')", block_result)
expected = 1.5 * (2 * 3.141592653589793 * 500 + 2 * 300 * 2000) / 1000000
assert abs(block_result["value"] - expected) < 1e-9
assert set(block_result["values"]) == {"S_V1", "S_V2", "S"}
